# Lecture 03 — ETL Pipelines i Python

PGR215 Data Collection and Analysis — Kristiania University College

## 1. Hva er ETL?

**ETL** = Extract, Transform, Load

1. **Extract**: Hent data fra ulike kilder (CSV, JSON, Excel, API)
2. **Transform**: Rens, valider, berik og standardiser data
3. **Load**: Lagre ferdig data til mål (CSV, database, warehouse)

I dette kurset bruker vi **pandas** som hovedverktøy for ETL i Python.

## 2. Extract — Lese data med Pandas

Pandas støtter mange filformater:

In [ ]:
import pandas as pd
import json
import os
import datetime

# === CSV ===
print("=== CSV ===")
df_sales = pd.read_csv('../04_data/sales_data.csv')
print(f"Shape: {df_sales.shape}")
display(df_sales.head(3))

# === JSON ===
print("\n=== JSON ===")
df_products = pd.read_json('../04_data/product_data.json')
print(f"Shape: {df_products.shape}")
display(df_products.head(3))

# === Excel ===
print("\n=== Excel ===")
try:
    df_customers = pd.read_excel('../04_data/customer_data.xlsx')
    print(f"Shape: {df_customers.shape}")
    display(df_customers.head(3))
except Exception as e:
    print(f"Feil: {e}")

In [ ]:
# Nyttige pandas-funksjoner for å forstå dataen
print("=== Info om sales_data ===")
print(f"Shape: {df_sales.shape}")
print(f"\nKolonner: {df_sales.columns.tolist()}")
print(f"\nDatatyper:\n{df_sales.dtypes}")
print(f"\nManglende verdier:\n{df_sales.isnull().sum()}")
print(f"\nStatistikk:")
display(df_sales.describe())

## 3. Transform — Rense og bearbeide data

Vanlige transformasjoner:

| Operasjon | Pandas-metode |
|-----------|---------------|
| Fjerne duplikater | `df.drop_duplicates()` |
| Håndtere manglende verdier | `df.dropna()`, `df.fillna()` |
| Endre datatyper | `pd.to_numeric()`, `pd.to_datetime()` |
| Standardisere tekst | `df[col].str.lower()`, `.str.strip()` |
| Rename kolonner | `df.rename(columns={...})` |
| Filtrere rader | `df[df[col] > verdi]` |
| Lage nye kolonner | `df['ny'] = df['a'] + df['b']` |
| Merge/join datasett | `pd.merge(df1, df2, on='key')` |

In [ ]:
# Transformer sales_data steg for steg
df = df_sales.copy()
print(f"Original: {df.shape}")

# 1. Fjern duplikater
before = len(df)
df = df.drop_duplicates()
print(f"\n1. Fjernet {before - len(df)} duplikater → {len(df)} rader")

# 2. Sjekk og håndter manglende verdier
print(f"\n2. Manglende verdier:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  Ingen manglende verdier!")

# 3. Standardiser kolonnenavn
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
print(f"\n3. Kolonner etter standardisering: {df.columns.tolist()}")

# 4. Vis resultatet
print(f"\nTransformert data:")
display(df.head())

In [ ]:
# Dato-transformasjoner
print("=== Dato-transformasjoner ===")
dato_eksempler = pd.DataFrame({
    'dato_str': ['2024-01-15', '15/02/2024', 'Mar 20, 2024', '2024.04.10']
})
print("Før:")
print(dato_eksempler['dato_str'].tolist())

# Konverter til datetime
dato_eksempler['dato'] = pd.to_datetime(dato_eksempler['dato_str'], format='mixed')
print(f"\nEtter pd.to_datetime():")
print(dato_eksempler['dato'].tolist())

# Trekk ut komponenter
dato_eksempler['ar'] = dato_eksempler['dato'].dt.year
dato_eksempler['maned'] = dato_eksempler['dato'].dt.month
dato_eksempler['ukedag'] = dato_eksempler['dato'].dt.day_name()
display(dato_eksempler)

In [ ]:
# Merge/join datasett
print("=== Merge to datasett ===")
ansatte = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'navn': ['Anna', 'Erik', 'Sara', 'Ole'],
    'dept_id': [10, 20, 10, 30]
})

avdelinger = pd.DataFrame({
    'dept_id': [10, 20, 30],
    'avdeling': ['IT', 'HR', 'Salg']
})

print("Ansatte:")
display(ansatte)
print("\nAvdelinger:")
display(avdelinger)

# Inner join
df_merged = pd.merge(ansatte, avdelinger, on='dept_id', how='inner')
print("\nEtter merge:")
display(df_merged)

## 4. Load — Lagre resultatet

Etter transformasjon lagrer vi data til ønsket format:

In [ ]:
# Lagre til ulike formater
df_result = df_merged.copy()

# CSV
output_csv = '../06_output/ansatte_med_avdeling.csv'
df_result.to_csv(output_csv, index=False)
print(f"Lagret CSV: {output_csv}")

# JSON
output_json = '../06_output/ansatte_med_avdeling.json'
df_result.to_json(output_json, orient='records', indent=2)
print(f"Lagret JSON: {output_json}")

# Verifiser
print(f"\nVerifisering — lest tilbake fra CSV:")
display(pd.read_csv(output_csv))

## 5. Logging — Spor ETL-prosessen

God praksis: Logg hvert steg i pipeline-en til en fil.

In [ ]:
# ETL med logging
def etl_med_logging(input_path, output_path, log_path):
    log_lines = []
    
    def log(msg):
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        log_lines.append(line)
        print(line)
    
    # EXTRACT
    log(f"EXTRACT: Leser {input_path}")
    df = pd.read_csv(input_path)
    log(f"EXTRACT: {len(df)} rader, {len(df.columns)} kolonner")
    
    # TRANSFORM
    log("TRANSFORM: Starter transformasjon")
    
    # Fjern duplikater
    before = len(df)
    df = df.drop_duplicates()
    log(f"TRANSFORM: Fjernet {before - len(df)} duplikater")
    
    # Håndter missing values
    missing = df.isnull().sum().sum()
    log(f"TRANSFORM: {missing} manglende verdier funnet")
    df = df.dropna()
    log(f"TRANSFORM: {len(df)} rader etter dropna()")
    
    # Standardiser kolonnenavn
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    log(f"TRANSFORM: Kolonner standardisert")
    
    # LOAD
    df.to_csv(output_path, index=False)
    log(f"LOAD: Lagret {len(df)} rader til {output_path}")
    
    # Skriv logg
    with open(log_path, 'w') as f:
        f.write('\n'.join(log_lines))
    log(f"LOG: Skrevet til {log_path}")
    
    return df

# Kjør pipeline
result = etl_med_logging(
    '../04_data/sales_data.csv',
    '../06_output/sales_clean.csv',
    '../06_output/etl_log.txt'
)

## 6. Komplett ETL-pipeline eksempel

La oss bygge en komplett pipeline som:
1. Leser fra flere kilder (CSV + JSON)
2. Renser og transformerer
3. Merger datasettene
4. Lagrer resultat + logg

In [ ]:
# Komplett ETL: Sales + Products
print("=" * 60)
print("  KOMPLETT ETL PIPELINE: Sales + Products")
print("=" * 60)

# EXTRACT
print("\n--- EXTRACT ---")
df_s = pd.read_csv('../04_data/sales_data.csv')
print(f"Sales: {df_s.shape}")

df_p = pd.read_json('../04_data/product_data.json')
print(f"Products: {df_p.shape}")

# TRANSFORM
print("\n--- TRANSFORM ---")

# Standardiser kolonnenavn
df_s.columns = [c.strip().lower().replace(' ', '_') for c in df_s.columns]
df_p.columns = [c.strip().lower().replace(' ', '_') for c in df_p.columns]
print(f"Sales kolonner: {df_s.columns.tolist()}")
print(f"Products kolonner: {df_p.columns.tolist()}")

# Fjern duplikater
df_s = df_s.drop_duplicates()
df_p = df_p.drop_duplicates()

# Vis første rader
print("\nSales (head):")
display(df_s.head(3))
print("\nProducts (head):")
display(df_p.head(3))

# LOAD
print("\n--- LOAD ---")
df_s.to_csv('../06_output/sales_transformed.csv', index=False)
df_p.to_csv('../06_output/products_transformed.csv', index=False)
print("Lagret til 06_output/")
print("\nPipeline fullført!")

## 7. Feilhåndtering i ETL

Robuste pipelines håndterer feil gracefully:

In [ ]:
# ETL med try/except
def safe_etl(input_path):
    try:
        # Extract
        if input_path.endswith('.csv'):
            df = pd.read_csv(input_path)
        elif input_path.endswith('.json'):
            df = pd.read_json(input_path)
        elif input_path.endswith('.xlsx'):
            df = pd.read_excel(input_path)
        else:
            raise ValueError(f"Ukjent filformat: {input_path}")
        
        print(f"OK: Leste {len(df)} rader fra {input_path}")
        return df
        
    except FileNotFoundError:
        print(f"FEIL: Filen {input_path} finnes ikke!")
        return None
    except ValueError as e:
        print(f"FEIL: {e}")
        return None
    except Exception as e:
        print(f"UVENTET FEIL: {type(e).__name__}: {e}")
        return None

# Test med ulike filer
for path in ['../04_data/sales_data.csv', '../04_data/finnes_ikke.csv', '../04_data/product_data.json']:
    safe_etl(path)
    print()

## Oppsummering

**Nøkkelkonsepter fra Lecture 03:**

1. ETL = Extract → Transform → Load
2. **Extract**: `pd.read_csv()`, `pd.read_json()`, `pd.read_excel()`
3. **Transform**: `drop_duplicates()`, `dropna()`, `fillna()`, `merge()`, `to_datetime()`
4. **Load**: `to_csv()`, `to_json()`, `to_parquet()`
5. Alltid logg ETL-prosessen
6. Bruk try/except for feilhåndtering
7. Bronze → Silver → Gold for datakvalitetslag